# Notebook 02 — Data Cleaning & ETL Pipeline

**Project:** Zomato Delivery Operations Analytics
**Sector:** Food Tech / Delivery Operations

###Objective
This notebook covers:
1. Loading the raw dataset
2. Fixing column name typos
3. Handling missing values
4. Fixing data type issues
5. Feature engineering
6. Saving cleaned dataset to data/processed/



In [1]:
#imports
import pandas as pd
import numpy as np


In [2]:
# Load the raw dataset
df = pd.read_csv('/content/Zomato_Dataset.csv')

print("Raw dataset loaded!")
print(f"Rows    : {df.shape[0]}")
print(f"Columns : {df.shape[1]}")

Raw dataset loaded!
Rows    : 45584
Columns : 20


In [3]:
# Fix typo in column name from 'Time_Orderd' to 'Time_Ordered'
df.rename(columns={'Time_Orderd': 'Time_Ordered'}, inplace=True)

print("Column name fixed!")
print(df.columns.tolist())

Column name fixed!
['ID', 'Delivery_person_ID', 'Delivery_person_Age', 'Delivery_person_Ratings', 'Restaurant_latitude', 'Restaurant_longitude', 'Delivery_location_latitude', 'Delivery_location_longitude', 'Order_Date', 'Time_Ordered', 'Time_Order_picked', 'Weather_conditions', 'Road_traffic_density', 'Vehicle_condition', 'Type_of_order', 'Type_of_vehicle', 'multiple_deliveries', 'Festival', 'City', 'Time_taken (min)']


In [4]:
# Fix typo in City column from 'Metropolitian' to 'Metropolitan'
df['City'] = df['City'].str.replace('Metropolitian', 'Metropolitan')

print("City typo fixed!")
print(df['City'].unique())

City typo fixed!
['Metropolitan' 'Urban' 'Semi-Urban' nan]


In [7]:
# Fill numerical columns with median
df['Delivery_person_Age'] = df['Delivery_person_Age'].fillna(df['Delivery_person_Age'].median())
df['Delivery_person_Ratings'] = df['Delivery_person_Ratings'].fillna(df['Delivery_person_Ratings'].median())
df['multiple_deliveries'] = df['multiple_deliveries'].fillna(df['multiple_deliveries'].median())

# Fill categorical columns with mode
df['Weather_conditions'] = df['Weather_conditions'].fillna(df['Weather_conditions'].mode()[0])
df['Road_traffic_density'] = df['Road_traffic_density'].fillna(df['Road_traffic_density'].mode()[0])
df['Festival'] = df['Festival'].fillna(df['Festival'].mode()[0])
df['City'] = df['City'].fillna(df['City'].mode()[0])
df['Time_Ordered'] = df['Time_Ordered'].fillna(df['Time_Ordered'].mode()[0])

print("Missing values handled!")
print(df.isnull().sum())

Missing values handled!
ID                             0
Delivery_person_ID             0
Delivery_person_Age            0
Delivery_person_Ratings        0
Restaurant_latitude            0
Restaurant_longitude           0
Delivery_location_latitude     0
Delivery_location_longitude    0
Order_Date                     0
Time_Ordered                   0
Time_Order_picked              0
Weather_conditions             0
Road_traffic_density           0
Vehicle_condition              0
Type_of_order                  0
Type_of_vehicle                0
multiple_deliveries            0
Festival                       0
City                           0
Time_taken (min)               0
dtype: int64


In [8]:
# Convert Order_Date to datetime
df['Order_Date'] = pd.to_datetime(df['Order_Date'], dayfirst=True)

# Convert Time columns to string (for time operations later)
df['Time_Ordered'] = df['Time_Ordered'].astype(str)
df['Time_Order_picked'] = df['Time_Order_picked'].astype(str)

print("Data types fixed!")
print(df.dtypes)

Data types fixed!
ID                                     object
Delivery_person_ID                     object
Delivery_person_Age                   float64
Delivery_person_Ratings               float64
Restaurant_latitude                   float64
Restaurant_longitude                  float64
Delivery_location_latitude            float64
Delivery_location_longitude           float64
Order_Date                     datetime64[ns]
Time_Ordered                           object
Time_Order_picked                      object
Weather_conditions                     object
Road_traffic_density                   object
Vehicle_condition                       int64
Type_of_order                          object
Type_of_vehicle                        object
multiple_deliveries                   float64
Festival                               object
City                                   object
Time_taken (min)                        int64
dtype: object


In [12]:
# Extract day, month, year from Order_Date
df['Order_Day'] = df['Order_Date'].dt.day
df['Order_Month'] = df['Order_Date'].dt.month
df['Order_Year'] = df['Order_Date'].dt.year
df['Day_of_Week'] = df['Order_Date'].dt.day_name()

# Extract order hour safely
df['Order_Hour'] = pd.to_datetime(df['Time_Ordered'], format='%H:%M', errors='coerce').dt.hour

print("New features created!")
print(df[['Order_Date', 'Order_Day', 'Order_Month', 'Order_Year', 'Day_of_Week', 'Order_Hour']].head())

New features created!
  Order_Date  Order_Day  Order_Month  Order_Year Day_of_Week  Order_Hour
0 2022-02-12         12            2        2022    Saturday        21.0
1 2022-02-13         13            2        2022      Sunday        14.0
2 2022-03-04          4            3        2022      Friday        17.0
3 2022-02-13         13            2        2022      Sunday         9.0
4 2022-02-14         14            2        2022      Monday        19.0


In [13]:
# Final shape and missing value check
print(f"Rows    : {df.shape[0]}")
print(f"Columns : {df.shape[1]}")
print()
print("Missing values:")
print(df.isnull().sum())

Rows    : 45584
Columns : 25

Missing values:
ID                                0
Delivery_person_ID                0
Delivery_person_Age               0
Delivery_person_Ratings           0
Restaurant_latitude               0
Restaurant_longitude              0
Delivery_location_latitude        0
Delivery_location_longitude       0
Order_Date                        0
Time_Ordered                      0
Time_Order_picked                 0
Weather_conditions                0
Road_traffic_density              0
Vehicle_condition                 0
Type_of_order                     0
Type_of_vehicle                   0
multiple_deliveries               0
Festival                          0
City                              0
Time_taken (min)                  0
Order_Day                         0
Order_Month                       0
Order_Year                        0
Day_of_Week                       0
Order_Hour                     4068
dtype: int64


In [14]:
# Save cleaned dataset to processed folder
df.to_csv('/content/Zomato_Cleaned.csv', index=False)

print("Cleaned dataset saved!")
print(f"Rows    : {df.shape[0]}")
print(f"Columns : {df.shape[1]}")

Cleaned dataset saved!
Rows    : 45584
Columns : 25


##Cleaning Summary

### Changes Made
1. Fixed column name typo — 'Time_Orderd' → 'Time_Ordered'
2. Fixed City typo — 'Metropolitian' → 'Metropolitan'
3. Filled numerical missing values with median
   - Delivery_person_Age     : 1,854 filled
   - Delivery_person_Ratings : 1,908 filled
   - multiple_deliveries     : 993 filled
4. Filled categorical missing values with mode
   - Weather_conditions      : 616 filled
   - Road_traffic_density    : 601 filled
   - Festival                : 228 filled
   - City                    : 1,200 filled
   - Time_Ordered            : 1,731 filled
5. Converted Order_Date to datetime format
6. Extracted new features
   - Order_Day, Order_Month, Order_Year
   - Day_of_Week
   - Order_Hour

### Output
- Cleaned file saved to : Zomato_Cleaned.csv
- Final Rows            : 45,584
- Final Columns         : 25

### Next Step
EDA and visualizations in 03_eda.ipynb